![DB Academy](https://files.training.databricks.com/binder/prod_main/get-started-with-databricks-for-data-warehousing-en_us-3.0.3/images/20261012T131742Z/Get Started with Databricks for Data Warehousing/Includes/images/db-academy.png)

<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">
<div style="background: #02A36F; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Capstone Lab</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Vintage Audio Quarterly Performance Review</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">Put the whole course together: query the star schema, build an optimized reporting table, govern your metrics with a metric view, and deliver the insight.</div>
</div>
</div>


## REQUIRED: SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

This notebook runs on Serverless compute. Before running any cells, confirm Serverless is selected in the compute dropdown at the top-right of this notebook. This course was developed and tested on <strong>Serverless environment version 5</strong>.
  </div>
</div>

### Setup
Run the cell below to configure your environment for this capstone. The setup script initializes your catalog and schema and makes sure the full star schema (`fact_orders` plus the `dim_*` tables) is in place, so your queries run against the correct context.


In [0]:
%run ./Includes/Classroom-Setup-Capstone

## The scenario

You're a data warehousing practitioner at **Vintage Audio Co.** The VP of Sales has asked for a **2024 quarterly performance review**: which product categories drive revenue, how each store is trending quarter over quarter, and one trusted set of numbers that every dashboard, Genie Agent, and BI tool can share.

This capstone has no new concepts. It asks you to combine what you practiced across the course into one short end-to-end flow:

| Task | What you'll do | Skill from |
|---|---|---|
| 1 | Confirm your starting point | Lessons 02-03 |
| 2 | Answer a business question against the star schema | Lessons 02, 08-09 |
| 3 | Materialize an **optimized** reporting table (`CLUSTER BY AUTO`) | Lessons 04-05 |
| 4 | Define a governed **metric view** | Lessons 10-11 |
| 5 | Deliver the insight with an inline **visualization** | Lesson 02 |

**Estimated time: ~20 minutes.**

**How to work each task.** The default path is **Genie Code**: open it inline with **Cmd/Ctrl + I** in the target cell (or use the full **Genie / Assistant** panel), give it the prompt provided, run the result, and expand the reference to compare. You can always write the SQL yourself — the reference solution is there whenever you want it.


### Task 1: Confirm your starting point

Before building anything, confirm the star schema is present. Run the cell below — it reports the row count of the fact table and each dimension in one result.


In [0]:
%sql
SELECT 'fact_orders'  AS table_name, COUNT(*) AS row_count FROM fact_orders
UNION ALL SELECT 'dim_customer', COUNT(*) FROM dim_customer
UNION ALL SELECT 'dim_product',  COUNT(*) FROM dim_product
UNION ALL SELECT 'dim_store',    COUNT(*) FROM dim_store
UNION ALL SELECT 'dim_date',     COUNT(*) FROM dim_date
ORDER BY row_count DESC;

You should see five rows: `fact_orders` = **297,998**, `dim_customer` = **5,000**, `dim_product` = **250**, `dim_store` = **30**, and `dim_date` = **1,461**. If any table is missing or `0`, re-run the setup cell above — its backfill rebuilds the model automatically.


### Task 2: Which categories drive revenue?

The VP's first question: **for 2024, how does each product category rank by revenue, and how many orders does each represent?** This joins `fact_orders` to `dim_product` (for `category`) and `dim_date` (to filter the year).

Return one row per category with **total revenue** (sum of `total_amount`, rounded to 2 decimals), **order count**, and **average order value** (revenue ÷ orders, rounded to 2 decimals), sorted by revenue descending.

**Genie Code prompt:**

<div style="position:relative; margin:6px 0;">
<button onclick="var t=document.getElementById('gp_6a7d222a').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:5px 11px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#EEF3F5; border-left:3px solid #1B5162; border-radius:6px; padding:12px 14px; margin:0; font-size:14px; white-space:pre-wrap;"><code id="gp_6a7d222a" style="white-space:pre-wrap; overflow-wrap:break-word;">Using fact_orders joined to dim_product and dim_date on their surrogate keys, show each product category with its total revenue (sum of total_amount rounded to 2 decimals), order count, and average order value (revenue divided by order count, rounded to 2 decimals) for the year 2024. Group by category and sort by revenue descending.</code></pre>
</div>

In [0]:
%sql
-- Use Genie Code (Cmd/Ctrl + I) with the prompt above, then run it.

<details>
<summary><strong>Here is the category revenue query</strong></summary>
<div style="position:relative; margin-top:8px;">
<button onclick="var t=document.getElementById('cb_14466253').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:6px 12px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#F9F7F4; border-radius:6px; padding:14px; margin:0; font-size:14px; overflow:auto;"><code id="cb_14466253">SELECT
  dp.category,
  ROUND(SUM(fo.total&#95;amount), 2)               AS revenue,
  COUNT(*)                                     AS order&#95;count,
  ROUND(SUM(fo.total&#95;amount) / COUNT(*), 2)    AS avg&#95;order&#95;value
FROM fact&#95;orders fo
JOIN dim&#95;product dp ON fo.product&#95;key = dp.product&#95;key
JOIN dim&#95;date    dd ON fo.date&#95;key    = dd.date&#95;key
WHERE dd.year = 2024
GROUP BY dp.category
ORDER BY revenue DESC;</code></pre>
</div>
</details>

You should see five rows, one per category, led by **Amplifiers** (≈ \$228.3M) and **Turntables** (≈ \$121.4M), then **Speakers** (≈ \$100.3M), **Accessories** (≈ \$45.2M), and **Cassette Decks** (≈ \$27.2M).

Notice the story in the `avg_order_value` column: **Accessories has the most orders by far (≈ 78,000) but the lowest revenue** — lots of small-ticket sales (≈ \$580 each), while Amplifiers move fewer units at a much higher value (≈ \$5,278 each). Volume and value are different questions, and the star schema lets you answer both from one join.


### Task 3: Build an optimized reporting table

The VP wants a **per-store, per-quarter** view of 2024 they can hand to the BI team without re-deriving it every time. Materialize it as a Unity Catalog managed table named **`store_quarterly_revenue`**, and enable **automatic liquid clustering** (`CLUSTER BY AUTO`) so Predictive Optimization keeps it laid out for however the BI team ends up filtering it — no partition columns or `ZORDER` to maintain.

The table should have one row per **store x quarter** for 2024, with the store's descriptive attributes and the quarter's totals: `store_id`, `store_name`, `region`, `state`, `year`, `quarter`, `order_count`, `total_revenue`.

**Genie Code prompt:**

<div style="position:relative; margin:6px 0;">
<button onclick="var t=document.getElementById('gp_5f775884').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:5px 11px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#EEF3F5; border-left:3px solid #1B5162; border-radius:6px; padding:12px 14px; margin:0; font-size:14px; white-space:pre-wrap;"><code id="gp_5f775884" style="white-space:pre-wrap; overflow-wrap:break-word;">Create a Unity Catalog managed table named store_quarterly_revenue with CLUSTER BY AUTO, populated from fact_orders joined to dim_store and dim_date. Include store_id, store_name, region, state, year, quarter, a count of orders as order_count, and the sum of total_amount rounded to 2 decimals as total_revenue, for the year 2024, grouped by store and quarter.</code></pre>
</div>

In [0]:
%sql
-- Use Genie Code with the prompt above, then run it.

<details>
<summary><strong>Here is the CREATE TABLE statement</strong></summary>
<div style="position:relative; margin-top:8px;">
<button onclick="var t=document.getElementById('cb_f25e0ced').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:6px 12px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#F9F7F4; border-radius:6px; padding:14px; margin:0; font-size:14px; overflow:auto;"><code id="cb_f25e0ced">CREATE OR REPLACE TABLE store&#95;quarterly&#95;revenue
CLUSTER BY AUTO AS
SELECT
  ds.store&#95;id,
  ds.store&#95;name,
  ds.region,
  ds.state,
  dd.year,
  dd.quarter,
  COUNT(*)                       AS order&#95;count,
  ROUND(SUM(fo.total&#95;amount), 2) AS total&#95;revenue
FROM fact&#95;orders fo
JOIN dim&#95;store ds ON fo.store&#95;key = ds.store&#95;key
JOIN dim&#95;date  dd ON fo.date&#95;key  = dd.date&#95;key
WHERE dd.year = 2024
GROUP BY ds.store&#95;id, ds.store&#95;name, ds.region, ds.state, dd.year, dd.quarter;</code></pre>
</div>
</details>

Now confirm the table was created at the grain you expect. Run the cell below.


In [0]:
%sql
SELECT COUNT(*) AS row_count FROM store_quarterly_revenue;

You should see **120** — exactly **30 stores × 4 quarters**. Every store sold in every quarter of 2024, so the grid is complete. Because you used `CLUSTER BY AUTO`, you didn't have to decide up front whether the BI team filters by `region`, `state`, or `quarter`; Predictive Optimization adapts the layout to the real query pattern over time.


### Task 4: Govern your metrics with a metric view

So far every query re-wrote `SUM(total_amount)` and its joins by hand. That's how numbers drift: two analysts define "revenue" slightly differently and now the dashboards disagree. Fix it by defining the measures **once** in a Unity Catalog **metric view** that every consumer — SQL Editor, dashboards, Genie Agents, alerts, and external BI — reads from.

Create a metric view named **`vw_quarterly_metrics`** on `fact_orders`, joined to `dim_store`, `dim_product`, and `dim_date`. Expose **dimensions** `region`, `category`, `year`, and `quarter`, and **measures** `total_revenue` (sum of `total_amount`), `order_count` (count of `order_id`), and `avg_order_value` (average of `total_amount`).

**Genie Code prompt:**

<div style="position:relative; margin:6px 0;">
<button onclick="var t=document.getElementById('gp_7a47de37').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:5px 11px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#EEF3F5; border-left:3px solid #1B5162; border-radius:6px; padding:12px 14px; margin:0; font-size:14px; white-space:pre-wrap;"><code id="gp_7a47de37" style="white-space:pre-wrap; overflow-wrap:break-word;">Create a Unity Catalog metric view named vw_quarterly_metrics using CREATE VIEW with METRICS LANGUAGE YAML. The source is fact_orders. Join dim_store on store_key, dim_product on product_key, and dim_date on date_key. Add dimensions region from dim_store, category from dim_product, and year and quarter from dim_date. Add measures total_revenue as SUM of total_amount, order_count as COUNT of order_id, and avg_order_value as AVG of total_amount.</code></pre>
</div>

In [0]:
%sql
-- Use Genie Code with the prompt above to generate the CREATE VIEW statement, then run it.

<details>
<summary><strong>Here is the full CREATE statement</strong></summary>
<div style="position:relative; margin-top:8px;">
<button onclick="var t=document.getElementById('cb_0fde460e').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:6px 12px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#F9F7F4; border-radius:6px; padding:14px; margin:0; font-size:14px; overflow:auto;"><code id="cb_0fde460e">CREATE OR REPLACE VIEW vw&#95;quarterly&#95;metrics
WITH METRICS LANGUAGE YAML AS $$
version: 1.1
source: fact&#95;orders
joins:
  - name: dim&#95;store
    source: dim&#95;store
    &quot;on&quot;: dim&#95;store.store&#95;key = source.store&#95;key
  - name: dim&#95;product
    source: dim&#95;product
    &quot;on&quot;: dim&#95;product.product&#95;key = source.product&#95;key
  - name: dim&#95;date
    source: dim&#95;date
    &quot;on&quot;: dim&#95;date.date&#95;key = source.date&#95;key
dimensions:
  - name: region
    expr: dim&#95;store.region
    display&#95;name: Region
  - name: category
    expr: dim&#95;product.category
    display&#95;name: Category
  - name: year
    expr: dim&#95;date.year
    display&#95;name: Year
  - name: quarter
    expr: dim&#95;date.quarter
    display&#95;name: Quarter
measures:
  - name: total&#95;revenue
    expr: SUM(source.total&#95;amount)
    display&#95;name: Total Revenue
  - name: order&#95;count
    expr: COUNT(source.order&#95;id)
    display&#95;name: Order Count
  - name: avg&#95;order&#95;value
    expr: AVG(source.total&#95;amount)
    display&#95;name: Average Order Value
$$;</code></pre>
</div>
</details>

Now confirm the metric view was registered in the `vintage_audio` schema. Run the cell below.


In [0]:
%sql
SHOW VIEWS LIKE 'vw_quarterly_metrics';

You should see one row listing `vw_quarterly_metrics`. If the result is empty, re-run the `CREATE` statement above and confirm your catalog and schema are set (the setup cell handles this).


### Task 5: Deliver the insight

Time to answer the headline question with the metric view instead of raw joins. Querying a metric view follows three rules from Lesson 10: no `SELECT *`, wrap every measure in `MEASURE()`, and use `GROUP BY ALL`. Notice how short the query is now — the joins and the definition of "revenue" live in the view.

Run the cell below to get **2024 revenue and order count by quarter**.


In [0]:
%sql
SELECT
  quarter,
  MEASURE(total_revenue)   AS revenue,
  MEASURE(order_count)     AS orders,
  MEASURE(avg_order_value) AS avg_order_value
FROM vw_quarterly_metrics
WHERE year = 2024
GROUP BY ALL
ORDER BY quarter;

You should see four rows — one per quarter — with revenue **climbing every quarter**: roughly **Q1 \$111.6M → Q2 \$122.2M → Q3 \$133.4M → Q4 \$155.2M**. That steady upward trend, straight from the governed metric definitions, is the story to bring to the VP.


#### Turn it into a chart

A table of four numbers is fine; a chart makes the trend obvious. Build an inline visualization from the result above (the same way you did in Lesson 02):

1. In the result panel of the cell above, click the **+** next to the **Table** tab and choose **Visualization**.
2. Set the **Visualization Type** to **Bar**.
3. Set the **X column** to `quarter` and the **Y column** to `revenue`.
4. Click **Save**. You now have a quarterly revenue bar chart you could drop straight into a dashboard.

That chart, backed by `vw_quarterly_metrics`, would show the *same* revenue number whether it's rendered here, in an AI/BI dashboard, in a Genie Agent answer, or in an external BI tool — because they all read the one definition you created in Task 4.


## Nice work — capstone complete
<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
<div style="color:#333;">
In this capstone you took a business request from raw star schema to a governed, visualized answer — the everyday loop of warehousing on Databricks:

<ul>
<li><strong>Queried the star schema</strong> to rank categories by revenue and expose the volume-vs-value story (Task 2).</li>
<li><strong>Materialized an optimized reporting table</strong> with <code>CLUSTER BY AUTO</code>, letting Predictive Optimization own the data layout (Task 3).</li>
<li><strong>Defined a metric view</strong> so <code>total_revenue</code>, <code>order_count</code>, and <code>avg_order_value</code> mean the same thing everywhere (Task 4).</li>
<li><strong>Delivered the insight</strong> as a short metric-view query and an inline chart the VP can act on (Task 5).</li>
</ul>

Everything you built — <code>store_quarterly_revenue</code> and <code>vw_quarterly_metrics</code> — lives in your governed <code>vintage_audio</code> schema, ready for a dashboard or a Genie Agent. The closing <strong>Summary and Next Steps</strong> notebook recaps the full course and points you toward where to go next.
</div>
</div>



&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>